# cSPARC Latent Space Comparison (Two Independent .cs Files)

Compares latent space embeddings from **two separate** cSPARC `.cs` particle files
(e.g. two different jobs, or two pre-split subsets that were each saved to their own `.cs` file).

Unlike the original `cSPARC_LatentSPACE.ipynb` (which loads a single combined `.cs` file and
splits it by index into OA/OEA), this notebook loads `PATH_A` and `PATH_B` independently, so the
two datasets do not need to be the same size or come from the same job.

Set the configuration in the cell below, then run the notebook top to bottom.

In [ ]:
import numpy as np
import re
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

# --- User configuration ---
PATH_A = Path("./csparc/jobA_particles.cs")
PATH_B = Path("./csparc/jobB_particles.cs")
LABEL_A = "Dataset A"
LABEL_B = "Dataset B"
OUTPUT_DIR = Path("comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

dataA = np.load(PATH_A)
dataB = np.load(PATH_B)

print(f"{LABEL_A}: {dataA.shape} loaded from {PATH_A}")
print(f"{LABEL_B}: {dataB.shape} loaded from {PATH_B}")


## Extract latent space vectors

Pulls the `components_mode_X/value` columns out of each structured array and stacks them into an n-dimensional vector per particle.

In [ ]:
def extract_latent_space_vectors(structured_array):
    """
    Extracts component values from different modes and combines them into vectors.

    Parameters:
        - structured_array: A structured NumPy array containing component data.

    Returns:
        An n-dimensional numpy ndarray where each row is a vector of components from each mode.
    """
    component_columns = [col for col in structured_array.dtype.names if re.match(r'components_mode_\d+/value', col)]

    if not component_columns:
        raise ValueError("No component columns found in the structured array.")

    components = [structured_array[col] for col in component_columns]
    return np.stack(components, axis=1)

z_A = extract_latent_space_vectors(dataA)
z_B = extract_latent_space_vectors(dataB)

print(f"{LABEL_A} latent vectors: {z_A.shape}")
print(f"{LABEL_B} latent vectors: {z_B.shape}")


## Vector magnitude (L2 norm) comparison

In [ ]:
def calculate_vector_magnitudes(vectors):
    """
    Calculates the magnitudes of vectors extracted from a NumPy array.

    Parameters:
        - vectors: An n-D numpy ndarray where each row is a vector.

    Returns:
        A 1D numpy ndarray where each element is the magnitude of the corresponding vector.
    """
    return np.linalg.norm(vectors, axis=1)

magnitudes_A = calculate_vector_magnitudes(z_A)
magnitudes_B = calculate_vector_magnitudes(z_B)

plt.figure(figsize=(20, 10))

plt.subplot(1, 2, 1)
plt.hist(magnitudes_A, bins=500, alpha=0.5, label=LABEL_A)
plt.hist(magnitudes_B, bins=500, alpha=0.5, label=LABEL_B)
plt.title('Histogram of Vector Magnitudes')
plt.xlabel('Magnitude')
plt.xlim(0, 200)
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
plt.violinplot([magnitudes_A, magnitudes_B], showmeans=True)
plt.title('ViolinPlot of Vector Magnitudes')
plt.xticks([1, 2], [LABEL_A, LABEL_B])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "magnitude_histogram.png")
plt.show()

t_stat, p_value = stats.ttest_ind(magnitudes_A, magnitudes_B)
print('T-test')
print('T-Statistic:', t_stat)
print('P-Value:', p_value)

u_stat, p_value = stats.mannwhitneyu(magnitudes_A, magnitudes_B, alternative='two-sided')
print('Mann-Whitney U-Test')
print('U-Statistic:', u_stat)
print('P-Value:', p_value)

f_stat, p_value = stats.f_oneway(magnitudes_A, magnitudes_B)
print('ANOVA Test')
print('F-Statistic:', f_stat)
print('P-Value:', p_value)

print(f'Mean {LABEL_A} Magnitude:', np.mean(magnitudes_A), 'SD', np.std(magnitudes_A))
print(f'Mean {LABEL_B} Magnitude:', np.mean(magnitudes_B), 'SD', np.std(magnitudes_B))


## UMAP embedding

Embeddings are cached to `OUTPUT_DIR` so re-running the notebook doesn't recompute them.

In [ ]:
import umap
import joblib

def perform_and_save_umap(data, filename):
    filename = Path(filename)
    if filename.exists():
        print(f"Loading UMAP embeddings from {filename}...")
        return joblib.load(filename)
    print(f"Performing UMAP and saving results to {filename}...")
    umap_reducer = umap.UMAP(n_components=2, metric='euclidean', min_dist=0.1, n_neighbors=50,
                              spread=1.5, learning_rate=0.5, negative_sample_rate=10,
                              init='pca', random_state=42)
    embedding = umap_reducer.fit_transform(data)
    joblib.dump(embedding, filename)
    return embedding

data_umap_A = perform_and_save_umap(z_A, OUTPUT_DIR / "umap_A.pkl")
data_umap_B = perform_and_save_umap(z_B, OUTPUT_DIR / "umap_B.pkl")


## t-SNE embedding

In [ ]:
from sklearn.manifold import TSNE

def perform_and_save_tsne(data, filename, perplexity=100, n_iter=1000):
    filename = Path(filename)
    if filename.exists():
        print(f"Loading t-SNE embeddings from {filename}...")
        return joblib.load(filename)
    print(f"Performing t-SNE and saving results to {filename}...")
    tsne = TSNE(perplexity=perplexity, n_iter=n_iter, random_state=42)
    embedding = tsne.fit_transform(data)
    joblib.dump(embedding, filename)
    return embedding

data_tsne_A = perform_and_save_tsne(z_A, OUTPUT_DIR / "tsne_A.pkl")
data_tsne_B = perform_and_save_tsne(z_B, OUTPUT_DIR / "tsne_B.pkl")


## PCA

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)  # Reducing to 3 dimensions for visualization
data_pca_A = pca.fit_transform(z_A)
data_pca_B = pca.fit_transform(z_B)


## UMAP plots

In [ ]:
umap_magnitudes_A = np.linalg.norm(data_umap_A, ord=2, axis=1)
umap_magnitudes_B = np.linalg.norm(data_umap_B, ord=2, axis=1)

plt.figure(figsize=(20, 30))

plt.subplot(3, 2, 1)
plt.title(f'UMAP of latent space {LABEL_A}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.scatter(data_umap_A[:, 0], data_umap_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")

plt.subplot(3, 2, 2)
plt.hexbin(data_umap_A[:, 0], data_umap_A[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'UMAP Hexbin Visualization of {LABEL_A}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 3)
plt.title(f'UMAP of latent space {LABEL_B}')
plt.scatter(data_umap_B[:, 0], data_umap_B[:, 1], alpha=0.9, label=LABEL_B, marker=".")
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 4)
plt.hexbin(data_umap_B[:, 0], data_umap_B[:, 1], gridsize=50, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'UMAP Hexbin Visualization of {LABEL_B}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')

plt.subplot(3, 2, 5)
plt.scatter(data_umap_A[:, 0], data_umap_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_umap_B[:, 0], data_umap_B[:, 1], alpha=0.1, label=LABEL_B, marker=".")
plt.title(f'UMAP of {LABEL_A} and {LABEL_B}')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.legend()

plt.subplot(3, 2, 6)
plt.hist(umap_magnitudes_A, bins=100, alpha=0.5, label=LABEL_A)
plt.hist(umap_magnitudes_B, bins=100, alpha=0.5, label=LABEL_B)
plt.title('Histogram of UMAP Vector Magnitudes')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "umap_visualizations.png")
plt.show()


## PCA plots

In [ ]:
plt.figure(figsize=(20, 20))

plt.subplot(2, 2, 1)
plt.scatter(data_pca_A[:, 0], data_pca_A[:, 1], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 0], data_pca_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 1st and 2nd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 1')

plt.subplot(2, 2, 2)
plt.scatter(data_pca_A[:, 0], data_pca_A[:, 2], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 0], data_pca_B[:, 2], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 1st and 3rd Principal Component')
plt.xlabel('Principal Component 0')
plt.ylabel('Principal Component 2')

plt.subplot(2, 2, 3)
plt.scatter(data_pca_A[:, 1], data_pca_A[:, 2], alpha=0.9, label=LABEL_A, marker=".")
plt.scatter(data_pca_B[:, 1], data_pca_B[:, 2], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title('PCA of 2nd and 3rd Principal Component')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_visualizations.png")
plt.show()


## t-SNE plots

In [ ]:
plt.figure(figsize=(20, 30))

plt.subplot(3, 2, 1)
plt.scatter(data_tsne_A[:, 0], data_tsne_A[:, 1], alpha=0.3, label=LABEL_A, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_A}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.subplot(3, 2, 2)
plt.hexbin(data_tsne_A[:, 0], data_tsne_A[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'tSNE Hexbin Visualization of {LABEL_A}')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

plt.subplot(3, 2, 3)
plt.scatter(data_tsne_B[:, 0], data_tsne_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_B}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.subplot(3, 2, 4)
plt.hexbin(data_tsne_B[:, 0], data_tsne_B[:, 1], gridsize=40, cmap='Oranges', mincnt=1)
cbar = plt.colorbar()
cbar.set_label('Counts')
plt.title(f'tSNE Hexbin Visualization of {LABEL_B}')
plt.xlabel('tSNE Component 1')
plt.ylabel('tSNE Component 2')

plt.subplot(3, 2, 5)
plt.scatter(data_tsne_A[:, 0], data_tsne_A[:, 1], alpha=0.3, label=LABEL_A, marker=".")
plt.scatter(data_tsne_B[:, 0], data_tsne_B[:, 1], alpha=0.3, label=LABEL_B, marker=".")
plt.legend()
plt.title(f't-SNE Visualization of {LABEL_A} and {LABEL_B}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tsne_visualizations.png")
plt.show()
